In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
idp_name_1 = 'FakeCAS'
idp_username_1 = None
idp_password_1 = None
idp_name_2 = 'FakeCAS'
idp_username_2 = None
idp_password_2 = None
project_name = 'TEST-WORKFLOW-EXEC-202601030618-統合管理者'
workflow_name = None
start_form_content = 'Test request content'
default_result_path = None
close_on_fail = False
transition_timeout = 30000
browser_type = 'chromium'
ignore_https_errors = False

In [ ]:
import tempfile

if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for manager ({idp_name_1})')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if idp_username_2 is None:
    idp_username_2 = input(prompt=f'Username for executor ({idp_name_2})')
if idp_password_2 is None:
    idp_password_2 = getpass(prompt=f'Password for {idp_username_2}@{idp_name_2}')

if project_name is None:
    project_name = datetime.now().strftime('TEST-WORKFLOW-%Y%m%d%H%M')
if workflow_name is None:
    workflow_name = input(prompt='Workflow template name')

# ワークフローの実行

- サブシステム名: アドオン
- ページ/アドオン: Workflow
- 機能分類: ワークフロー実行
- シナリオ名: ワークフローの開始と承認（単一/複数ユーザー対応）
- 用意するテストデータ: アカウント(user1: manager/プロジェクト管理者, user2: executor/研究者)、プロジェクト名、ワークフローテンプレート名
- 事前条件: テンプレート登録が完了していること
- 備考: user1とuser2が同一ユーザーの場合は単一ユーザーテスト、異なる場合は複数ユーザーテストとなる

In [ ]:
work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

## 新規タブを開き、GRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await page.locator('//button[text() = "同意する"]').click()
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step, new_page=True)

## manager (user1) のログイン情報を用いてGakuNin RDMにログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクト一覧に指定されたタイトルのプロジェクトがない場合、プロジェクトを作成する

プロジェクト一覧に当該プロジェクト名が表示されていない場合、「新規プロジェクト作成」をクリックし、その名前を入力、「作成」をクリックする。

In [ ]:
async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)
    await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から対象プロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンを選択」のパネル内「Workflow」の行の「有効にする」をクリックする

「Workflowアドオン規約」のダイアログが表示されること

In [ ]:
async def _step(page):
    enable_button = page.locator('//div[@full_name = "Workflow"]//a[text() = "有効にする"]')
    if await enable_button.count():
        await enable_button.click()
        await expect(page.locator('//button[@data-bb-handler = "confirm"]')).to_be_visible(timeout=transition_timeout)
    else:
        print('Workflow addon is already enabled for this project')

await run_pw(_step)

## 「確認」をクリックする

「アドオンを構成」のパネル内に「Workflow」の行が追加されること

In [ ]:
async def _step(page):
    confirm_button = page.locator('//button[@data-bb-handler = "confirm"]')
    if await confirm_button.count():
        await confirm_button.click()
    await expect(page.locator('#activate-workflow-select')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ワークフローテンプレート」ドロップダウンからテンプレートを選択する

指定したテンプレートが選択されること

In [ ]:
async def _step(page):
    await page.locator('#activate-workflow-select').scroll_into_view_if_needed()
    option = page.locator(f'#activate-workflow-select option:has-text("{workflow_name}")')
    value = await option.get_attribute('value')
    await page.locator('#activate-workflow-select').select_option(value=value)

await run_pw(_step)

## 「有効化」をクリックする

「ワークフロー権限の付与」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.locator('#activationsPanel').locator('//button[.//span[contains(text(), "有効化")]]').click()
    await expect(page.locator("#activateTokenPermissionModal").locator('//h4[contains(text(), "ワークフロー権限の付与")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「権限を付与して有効化」をクリックする

テンプレートが有効化され、有効化済みテンプレート一覧に表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[contains(text(), "権限を付与して有効化")]').click()
    await expect(page.locator(f'#activationsPanel').locator(f'//td//strong[text()="{workflow_name}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクト名をクリックしてプロジェクトダッシュボードに戻る

プロジェクトダッシュボードが表示され、ワークフローパネルにテンプレートリンクが表示されること

In [ ]:
project_url = None

async def _step(page):
    global project_url
    await page.locator(f'//a[contains(@class, "project-title") and normalize-space(text()) = "{project_name}"]').click()
    await expect(page.locator('#workflow-dashboard')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'#workflow-dashboard').locator(f'//a[contains(@class, "btn-primary") and contains(text(), "{workflow_name}")]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)

## (複数ユーザーの場合) managerをログアウトする

ログイン画面が表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping logout')
        return
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) プロジェクトURL/workflowにアクセスしexecutorとしてログインする

「許可が必要です」画面が表示され、「アクセス権をリクエスト」ボタンが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping executor login for access request')
        return
    await page.goto(project_url.rstrip('/') + '/workflow')
    await grdm.login(page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout)
    await expect(page.locator('//button[text() = "アクセス権をリクエスト"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) 「アクセス権をリクエスト」をクリックする

「アクセス権がリクエストされました」ボタンが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping access request')
        return
    await page.locator('//button[text() = "アクセス権をリクエスト"]').click()
    await expect(page.locator('//button[text() = "アクセス権がリクエストされました"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) executorをログアウトしプロジェクトURLを開きmanagerとしてログインする

プロジェクトダッシュボードが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping user switch for access approval')
        return
    await grdm.logout(page, idp_name_2, transition_timeout=transition_timeout)
    await page.goto(project_url)
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) 「メンバー」をクリックする

「アクセスのリクエスト」セクションが表示され、executorからのリクエストが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping access approval')
        return
    await page.locator('#projectSubnav').get_by_role('link', name='メンバー').click()
    await expect(page.locator('//h3[contains(text(), "アクセスのリクエスト")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('.request-accept-button')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) 権限を「読込み/書込み」に変更し「追加」をクリックする

リクエストが承認され、「アクセスのリクエスト」セクションからexecutorの行が消えること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping access approval action')
        return
    await page.locator('#manageAccessRequests .permissions select').select_option('write')
    await page.locator('.request-accept-button').click()
    await expect(page.locator('#manageAccessRequests .permissions select')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) managerをログアウトしプロジェクトURLを開きexecutorとしてログインする

プロジェクトダッシュボードが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping user switch to executor')
        return
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)
    await page.goto(project_url)
    await grdm.login(page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (executor) ワークフローパネルのテンプレートリンクをクリックする

ワークフロー開始フォームが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'#workflow-dashboard').locator(f'//a[contains(@class, "btn-primary") and contains(text(), "{workflow_name}")]').click()
    await expect(page.locator('#workflow-field-confirm_checkbox')).to_be_visible(timeout=transition_timeout)
    await asyncio.sleep(3)

await run_pw(_step)

## (executor) 「内容を確認しました」をチェックし「申請内容」に値を入力する

入力内容が反映されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-field-confirm_checkbox').check()
    await page.locator('#workflow-field-request_content').fill(start_form_content)
    await expect(page.locator('#workflow-field-confirm_checkbox')).to_be_checked(timeout=transition_timeout)
    await expect(page.locator('#workflow-field-request_content')).to_have_value(start_form_content, timeout=transition_timeout)

await run_pw(_step)

## (executor) 「ワークフローを開始」をクリックする

単一ユーザーの場合はタスク一覧に「開く」ボタンが表示されること。複数ユーザーの場合はタスクが表示されないこと。

In [ ]:
async def _step(page):
    await page.locator('//button[@data-test-workflow-start-button]').click()
    await expect(page.locator('//*[@data-test-workflow-submit-success]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 10秒待ち、プロジェクトURLを開きコメントボタンをクリックする

コメントパネルが開き、申請内容を含む通知コメントが表示されること
(10秒はワークフローが開始してコメント送信が完了するまでの待機時間)

In [ ]:
async def _step(page):
    await asyncio.sleep(10)
    await page.goto(project_url)
    await expect(page.locator('.fa-comments-o')).to_be_visible(timeout=transition_timeout)
    await page.locator('.fa-comments-o').click()
    await expect(page.locator(f'//*[contains(text(), "{start_form_content}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) executorをログアウトしプロジェクトURLを開きmanagerとしてログインする

プロジェクトダッシュボードに承認タスクの「開く」ボタンが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
importlib.reload(scripts.grdm)
from scripts import grdm

async def _step(page):
    if idp_username_1 == idp_username_2:
        await page.reload()
        await page.locator('#workflow-dashboard').locator('.fa-pencil').scroll_into_view_if_needed()
        print('Single user mode: skipping user switch to manager for approval')
        return
    await grdm.logout(page, idp_name_2, transition_timeout=transition_timeout)
    await page.goto(project_url)
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)
    await page.locator('#workflow-dashboard').locator('.fa-pencil').scroll_into_view_if_needed()

await run_pw(_step)

## (manager) 承認タスクの「開く」をクリックする

審査フォームダイアログが表示され、executorが入力した申請内容が表示されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-dashboard').locator('.fa-pencil').click()
    await expect(page.locator('#workflow-field-approval_result')).to_be_visible(timeout=transition_timeout)
    # executorが入力した申請内容が表示されていることを確認
    await expect(page.locator(f'//*[contains(text(), "{start_form_content}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (manager) 審査フォームで「再提出」を選択し「コメント」を入力する

入力内容が反映されること

In [ ]:
resubmit_comment = 'Please revise and resubmit'

async def _step(page):
    option = page.locator('#workflow-field-approval_result option:has-text("再提出")')
    value = await option.get_attribute('value')
    await page.locator('#workflow-field-approval_result').select_option(value=value)
    await page.locator('#workflow-field-review_comment').fill(resubmit_comment)

await run_pw(_step)

## (manager) 「送信」をクリックする

単一ユーザーの場合は再申請タスクの「開く」ボタンが表示されること。複数ユーザーの場合はタスクが表示されないこと。

In [ ]:
async def _step(page):
    await page.locator('//button[@data-test-task-dialog-submit]').click()
    if idp_username_1 == idp_username_2:
        # 単一ユーザー: executorの再申請タスクが表示される
        await expect(page.locator('//h4[text() = "Resubmit Application"]')).to_be_visible(timeout=transition_timeout)
    else:
        # 複数ユーザー: タスクは表示されない（executorに割り当てられている）
        await expect(page.locator('//button[@data-test-task-dialog-submit]')).not_to_be_visible(timeout=transition_timeout)
        await expect(page.locator('//button[@data-test-workflow-task-open]')).to_have_count(0, timeout=transition_timeout)
    await asyncio.sleep(2)

await run_pw(_step)

## (複数ユーザーの場合) managerをログアウトしプロジェクトURLを開きexecutorとしてログインする

再申請タスクの「開く」ボタンが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping user switch to executor for resubmit')
        return
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)
    await page.goto(project_url)
    await grdm.login(page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (単一ユーザーの場合はプロジェクトURLを開き) コメントボタンをクリックする

コメントパネルが開き、再提出理由を含む通知コメントが表示されること

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        # 単一ユーザー: プロジェクトURLに移動
        await page.goto(project_url)
    await expect(page.locator('.fa-comments-o')).to_be_visible(timeout=transition_timeout)
    await page.locator('.fa-comments-o').click()
    await expect(page.locator(f'//*[contains(text(), "{resubmit_comment}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (executor) 再申請タスクの「開く」をクリックする

再申請フォームダイアログが表示されること

In [ ]:
async def _step(page):
    await page.reload()
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)
    await page.locator('#workflow-dashboard').locator('.fa-pencil').click()
    await expect(page.locator('#workflow-field-confirm_checkbox')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (executor) 「申請内容」に再提出メッセージを入力し「内容を確認しました」をチェックして「送信」をクリックする

単一ユーザーの場合は承認タスクの「開く」ボタンが表示されること。複数ユーザーの場合はタスクが表示されないこと。

In [ ]:
resubmit_content = 'Revised and resubmitted'

async def _step(page):
    await page.locator('#workflow-field-request_content').fill(resubmit_content)
    await page.locator('#workflow-field-confirm_checkbox').check()
    await page.locator('//button[@data-test-task-dialog-submit]').click()
    if idp_username_1 == idp_username_2:
        # 単一ユーザー: managerの承認タスクが表示される
        await expect(page.locator('//h4[text() = "Review Application"]')).to_be_visible(timeout=transition_timeout)
    else:
        # 複数ユーザー: タスクは表示されない（managerに割り当てられている）
        await expect(page.locator('//button[@data-test-task-dialog-submit]')).not_to_be_visible(timeout=transition_timeout)
        await expect(page.locator('//button[@data-test-workflow-task-open]')).to_have_count(0, timeout=transition_timeout)
    await asyncio.sleep(2)

await run_pw(_step)

## (複数ユーザーの場合) executorをログアウトしプロジェクトURLを開きmanagerとしてログインする

プロジェクトダッシュボードに承認タスクの「開く」ボタンが表示されること。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping user switch to manager for final approval')
        return
    await grdm.logout(page, idp_name_2, transition_timeout=transition_timeout)
    await page.goto(project_url)
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (manager) 承認タスクの「開く」をクリックする

審査フォームダイアログが表示され、executorが再入力した申請内容が表示されること

In [ ]:
async def _step(page):
    if idp_username_1 != idp_username_2:
        await page.locator('#workflow-dashboard').locator('.fa-pencil').click()
    await expect(page.locator('#workflow-field-approval_result')).to_be_visible(timeout=transition_timeout)
    # executorが再入力した申請内容が表示されていることを確認
    await expect(page.locator(f'//*[contains(text(), "{resubmit_content}")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (manager) 審査フォームで「承認」を選択し「コメント」を入力して「送信」をクリックする

ワークフローが完了し、タスク一覧にタスクが表示されないこと

In [ ]:
async def _step(page):
    option = page.locator('#workflow-field-approval_result option:has-text("承認")')
    value = await option.get_attribute('value')
    await page.locator('#workflow-field-approval_result').select_option(value=value)
    await page.locator('#workflow-field-review_comment').fill('Approved after revision')
    await page.locator('//button[@data-test-task-dialog-submit]').click()
    await expect(page.locator('//button[@data-test-task-dialog-submit]')).not_to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//button[@data-test-workflow-task-open]')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## (複数ユーザーの場合) managerをログアウトしプロジェクトURLを開きexecutorとしてログインする

プロジェクトダッシュボードが表示され、ワークフロータスクが存在しないこと。単一ユーザーの場合はスキップされる。

In [ ]:
async def _step(page):
    if idp_username_1 == idp_username_2:
        print('Single user mode: skipping executor verification')
        return
    await grdm.logout(page, idp_name_1, transition_timeout=transition_timeout)
    await page.goto(project_url)
    await grdm.login(page, idp_name_2, idp_username_2, idp_password_2, transition_timeout=transition_timeout)
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('#workflow-dashboard').locator('.fa-pencil')).to_have_count(0, timeout=transition_timeout)

await run_pw(_step)

## 後始末

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)

In [ ]:
!rm -fr {work_dir}